In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Test").master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/11 11:07:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.csv("data/raw/flight_data.csv", header=True, inferSchema=True)

ERROR:root:KeyboardInterrupt while sending command.             (48 + 16) / 232]
Traceback (most recent call last):
  File "/home/aryan-bodhe/Desktop/VSCode/IITH/Business Python/env/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/aryan-bodhe/Desktop/VSCode/IITH/Business Python/env/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [13]:
from pyspark.sql.functions import split, explode, col

# Collect unique codes from startingAirport and destinationAirport
direct = (
    df.select(col("startingAirport").alias("airport"))
    .union(df.select(col("destinationAirport").alias("airport")))
)

# Split the ||−delimited segment columns and explode into rows
segment_cols = ["segmentsArrivalAirportCode", "segmentsDepartureAirportCode"]
for c in segment_cols:
    direct = direct.union(
        df.select(explode(split(col(c), r"\|\|")).alias("airport"))
    )

unique_airport_count = direct.distinct().count()
print(f"Total unique airport codes across all fields: {unique_airport_count}")


Total unique airport codes across all fields: 228


In [14]:
airmap = spark.read.csv("data/airportMappings.csv", header=True, inferSchema=True)

unique_airports_df = direct.distinct().withColumnRenamed("airport", "AirportCode")

# Anti-join: codes in flight data but NOT in the mapping file
missing_airports = unique_airports_df.join(airmap, on="AirportCode", how="left_anti").orderBy("AirportCode")

print(f"Airport codes not in mapping file: {missing_airports.count()}")
missing_airports.show(missing_airports.count(), truncate=False)


Airport codes not in mapping file: 40


+-----------+
|AirportCode|
+-----------+
|ABE        |
|ACY        |
|AEX        |
|AMA        |
|ASE        |
|BIS        |
|BMI        |
|BOG        |
|CMI        |
|CNY        |
|CRW        |
|CUN        |
|DAB        |
|ECP        |
|ESC        |
|FAR        |
|GYE        |
|HNL        |
|HYA        |
|IDA        |
|ITH        |
|IWD        |
|KOA        |
|LBB        |
|LIH        |
|LWB        |
|MTJ        |
|OGG        |
|OGS        |
|ORH        |
|OTH        |
|PRC        |
|SBN        |
|SHD        |
|SLN        |
|STS        |
|STT        |
|TRI        |
|VPS        |
|YUM        |
+-----------+



In [15]:
missing_airports.coalesce(1).write.mode("overwrite").option("header", True).csv("newmaps_tmp")

import os, shutil, glob

# PySpark writes to a folder; extract the single part file and rename it
part_file = glob.glob("newmaps_tmp/part-*.csv")[0]
shutil.copy(part_file, "newmaps.csv")
shutil.rmtree("newmaps_tmp")

print("Saved to newmaps.csv")


Saved to newmaps.csv


In [60]:
df.columns

['legId',
 'searchDate',
 'flightDate',
 'startingAirport',
 'destinationAirport',
 'fareBasisCode',
 'travelDuration',
 'elapsedDays',
 'isBasicEconomy',
 'isRefundable',
 'isNonStop',
 'baseFare',
 'totalFare',
 'seatsRemaining',
 'totalTravelDistance',
 'segmentsDepartureTimeEpochSeconds',
 'segmentsDepartureTimeRaw',
 'segmentsArrivalTimeEpochSeconds',
 'segmentsArrivalTimeRaw',
 'segmentsArrivalAirportCode',
 'segmentsDepartureAirportCode',
 'segmentsAirlineName',
 'segmentsAirlineCode',
 'segmentsEquipmentDescription',
 'segmentsDurationInSeconds',
 'segmentsDistance',
 'segmentsCabinCode']

In [61]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [ ]:
from pyspark.sql.functions import col

df = df.withColumn(
    "segmentsDepartureTimeEpochSeconds",
    col("segmentsDepartureTimeEpochSeconds")
).withColumn(
    "segmentsArrivalTimeEpochSeconds",
    col("segmentsArrivalTimeEpochSeconds")
).withColumn(
    "segmentsDurationInSeconds",
    col("segmentsDurationInSeconds")
).withColumn(
    "segmentsDistance",
    col("segmentsDistance")
)

In [57]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: long (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: long (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: strin

In [62]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [63]:
df.select("travelDuration").show(5)

+--------------+
|travelDuration|
+--------------+
|       PT2H29M|
|       PT2H30M|
|       PT2H30M|
|       PT2H32M|
|       PT2H34M|
+--------------+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import *

df = df.withColumn(
    "hours",
    regexp_extract("travelDuration", r"PT(\d+)H", 1)
).withColumn(
    "minutes",
    regexp_extract("travelDuration", r"H(\d+)M", 1)
)

df = df.withColumn(
    "travelDurationMinutes",
    col("hours") * 60 + col("minutes")
).drop("hours", "minutes")


In [65]:
df.select("travelDurationMinutes").show(5)

+---------------------+
|travelDurationMinutes|
+---------------------+
|                  149|
|                  150|
|                  150|
|                  152|
|                  154|
+---------------------+
only showing top 5 rows


In [66]:
from pyspark.sql.functions import size

df = df.withColumn(
    "numStops",
    size(split("segmentsDepartureTimeRaw", r"\|\|")) - 1
)

In [67]:
df.select("numStops").show(20)

+--------+
|numStops|
+--------+
|       0|
|       0|
|       0|
|       0|
|       0|
|       0|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       1|
|       0|
|       1|
|       1|
|       1|
+--------+
only showing top 20 rows


In [74]:
from pyspark.sql.functions import split

df = df.withColumn(
    "firstDepartureRaw",
    split("segmentsDepartureTimeRaw", r"\|\|")[0]
)

from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "firstDepartureTimestamp",
    to_timestamp("firstDepartureRaw")
)

df = df.withColumn(
    "lastArrivalTimestamp",
    to_timestamp(
        element_at(
            split("segmentsArrivalTimeRaw", r"\|\|"),
            -1
        )
    )
)

In [75]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [76]:
df = df.drop(
    "firstDepartureRaw",
    "lastArrivalRaw",
    "hours",
    "minutes",
    "departureTimesArray"
)

In [77]:
df.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [ ]:
df.select(
    "searchDate",
    "flightDate", 
    "startingAirport", 
    "destinationAirport", 
    "travelDuration",
    "elapsedDays",
    "isBasicEconomy",
    "isRefundable",
    "isNonStop",
    "baseFare",
    "totalFare",
    "seatsRemaining",
    "totalTravelDistance",
    
)

{"ts": "2026-03-03 09:39:05.078", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`legId`, `baseFare`, `numStops`, `isNonStop`, `totalFare`]. SQLSTATE: 42703", "context": {"file": "jdk.internal.reflect.GeneratedMethodAccessor66.invoke(Unknown Source)", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o252.select.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`legId`, `baseFare`, `numStops`, `isNonStop`, `totalFare`]. SQLSTATE: 42703;\n'Project [searchDate#23, flightDate#24, startingAirport#25, destinationAirport#26, ']\n+- Project [legId#22, searchDate#23, flightDate#24

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `` cannot be resolved. Did you mean one of the following? [`legId`, `baseFare`, `numStops`, `isNonStop`, `totalFare`]. SQLSTATE: 42703;
'Project [searchDate#23, flightDate#24, startingAirport#25, destinationAirport#26, ']
+- Project [legId#22, searchDate#23, flightDate#24, startingAirport#25, destinationAirport#26, fareBasisCode#27, travelDuration#28, elapsedDays#29, isBasicEconomy#30, isRefundable#31, isNonStop#32, baseFare#33, totalFare#34, seatsRemaining#35, totalTravelDistance#36, segmentsDepartureTimeEpochSeconds#268L, segmentsDepartureTimeRaw#38, segmentsArrivalTimeEpochSeconds#269L, segmentsArrivalTimeRaw#40, segmentsArrivalAirportCode#41, segmentsDepartureAirportCode#42, segmentsAirlineName#43, segmentsAirlineCode#44, segmentsEquipmentDescription#45, segmentsDurationInSeconds#270, ... 6 more fields]
   +- Project [legId#22, searchDate#23, flightDate#24, startingAirport#25,

In [78]:
df.select("lastArrivalTimestamp").show(10)

+--------------------+
|lastArrivalTimestamp|
+--------------------+
| 2022-04-18 00:56:00|
| 2022-04-17 18:30:00|
| 2022-04-17 23:35:00|
| 2022-04-18 02:01:00|
| 2022-04-17 22:03:00|
| 2022-04-17 22:53:00|
| 2022-04-17 22:02:00|
| 2022-04-17 23:08:00|
| 2022-04-17 22:02:00|
| 2022-04-17 23:08:00|
+--------------------+
only showing top 10 rows


In [79]:
df.show(1)


+--------------------+----------+----------+---------------+------------------+-------------+--------------+-----------+--------------+------------+---------+--------+---------+--------------+-------------------+---------------------------------+------------------------+-------------------------------+----------------------+--------------------------+----------------------------+-------------------+-------------------+----------------------------+-------------------------+----------------+-----------------+---------------------+--------+-----------------------+--------------------+
|               legId|searchDate|flightDate|startingAirport|destinationAirport|fareBasisCode|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|baseFare|totalFare|seatsRemaining|totalTravelDistance|segmentsDepartureTimeEpochSeconds|segmentsDepartureTimeRaw|segmentsArrivalTimeEpochSeconds|segmentsArrivalTimeRaw|segmentsArrivalAirportCode|segmentsDepartureAirportCode|segmentsAirlineName|segmentsA

In [80]:
df = df.repartition(16)

In [81]:
df.write.mode("overwrite").parquet("flights_data_1")

26/03/03 11:27:29 ERROR Executor: Exception in task 10.0 in stage 28.0 (TID 531)
org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value '' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)

	at org.apache.spark.sql.errors.QueryExecutionErrors$.invalidInputInCastToNumberError(QueryExecutionErrors.scala:147)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.withException(UTF8StringUtils.scala:51)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils$.toIntExact(UTF8StringUtils.scala:34)
	at org.apache.spark.sql.catalyst.util.UTF8StringUtils.toIntExact(UTF8StringUtils.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(

NumberFormatException: [CAST_INVALID_INPUT] The value '' of the type "STRING" cannot be cast to "INT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


In [84]:
from pyspark.sql.functions import col, sum

null_counts = df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])


+-----+----------+----------+---------------+------------------+-------------+--------------+-----------+--------------+------------+---------+--------+---------+--------------+-------------------+---------------------------------+------------------------+-------------------------------+----------------------+--------------------------+----------------------------+-------------------+-------------------+----------------------------+-------------------------+----------------+-----------------+
|legId|searchDate|flightDate|startingAirport|destinationAirport|fareBasisCode|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|baseFare|totalFare|seatsRemaining|totalTravelDistance|segmentsDepartureTimeEpochSeconds|segmentsDepartureTimeRaw|segmentsArrivalTimeEpochSeconds|segmentsArrivalTimeRaw|segmentsArrivalAirportCode|segmentsDepartureAirportCode|segmentsAirlineName|segmentsAirlineCode|segmentsEquipmentDescription|segmentsDurationInSeconds|segmentsDistance|segmentsCabinCode|
+---

In [85]:
null_counts.show(vertical=True)

-RECORD 0------------------------------------
 legId                             | 0       
 searchDate                        | 0       
 flightDate                        | 0       
 startingAirport                   | 0       
 destinationAirport                | 0       
 fareBasisCode                     | 0       
 travelDuration                    | 0       
 elapsedDays                       | 0       
 isBasicEconomy                    | 0       
 isRefundable                      | 0       
 isNonStop                         | 0       
 baseFare                          | 0       
 totalFare                         | 0       
 seatsRemaining                    | 0       
 totalTravelDistance               | 6094532 
 segmentsDepartureTimeEpochSeconds | 0       
 segmentsDepartureTimeRaw          | 0       
 segmentsArrivalTimeEpochSeconds   | 0       
 segmentsArrivalTimeRaw            | 0       
 segmentsArrivalAirportCode        | 0       
 segmentsDepartureAirportCode     

In [88]:
df.select("legId").count()

82138753

In [89]:
df_no_null = df.dropna(subset=["totalTravelDistance", "segmentsEquipmentDescription"])

In [102]:
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

string_cols = [f.name for f in df_no_null.schema.fields if isinstance(f.dataType, StringType)]

for c in string_cols:
    df_no_null = df_no_null.filter(trim(col(c)) != "")

In [103]:
df_no_null.select("travelDuration").show(100)

+--------------+
|travelDuration|
+--------------+
|       PT2H29M|
|       PT2H30M|
|       PT2H30M|
|       PT2H32M|
|       PT2H34M|
|       PT4H12M|
|       PT5H18M|
|       PT5H32M|
|       PT6H38M|
|       PT4H46M|
|       PT5H45M|
|       PT5H59M|
|       PT7H18M|
|       PT8H10M|
|       PT4H17M|
|       PT4H36M|
|       PT4H45M|
|        PT6H2M|
|       PT6H14M|
|       PT9H46M|
|       PT11H1M|
|      PT11H17M|
|       PT6H17M|
|       PT6H25M|
|       PT2H32M|
|       PT2H33M|
|       PT2H35M|
|       PT4H40M|
|       PT5H25M|
|       PT5H26M|
|       PT6H25M|
|       PT4H23M|
|        PT1H8M|
|       PT1H10M|
|       PT1H11M|
|       PT1H11M|
|       PT1H13M|
|       PT1H14M|
|       PT1H14M|
|       PT1H17M|
|       PT1H21M|
|       PT1H22M|
|       PT1H26M|
|       PT1H30M|
|      PT10H57M|
|        PT5H6M|
|       PT5H52M|
|       PT8H28M|
|       PT9H48M|
|      PT12H11M|
|       PT8H48M|
|       PT1H26M|
|         PT12H|
|       PT3H12M|
|       PT3H14M|
|       PT3H17

In [104]:
df_no_null.select("legId").count()

74754290

In [107]:
from pyspark.sql.functions import regexp_extract, col, when, lit

df_no_null_fixed_duration = df_no_null.withColumn(
    "travelDurationMinutes",
    (
        when(
            regexp_extract("travelDuration", r"PT(\d+)H", 1) != "",
            regexp_extract("travelDuration", r"PT(\d+)H", 1).cast("int")
        ).otherwise(lit(0)) * 60
        +
        when(
            regexp_extract("travelDuration", r"H(\d+)M", 1) != "",
            regexp_extract("travelDuration", r"H(\d+)M", 1).cast("int")
        ).otherwise(lit(0))
    )
)

In [108]:
df_no_null_fixed_duration.select("travelDurationMinutes").show(100)

+---------------------+
|travelDurationMinutes|
+---------------------+
|                  149|
|                  150|
|                  150|
|                  152|
|                  154|
|                  252|
|                  318|
|                  332|
|                  398|
|                  286|
|                  345|
|                  359|
|                  438|
|                  490|
|                  257|
|                  276|
|                  285|
|                  362|
|                  374|
|                  586|
|                  661|
|                  677|
|                  377|
|                  385|
|                  152|
|                  153|
|                  155|
|                  280|
|                  325|
|                  326|
|                  385|
|                  263|
|                   68|
|                   70|
|                   71|
|                   71|
|                   73|
|                   74|
|               

In [111]:
df_no_null_fixed_duration.show(1)

+--------------------+----------+----------+---------------+------------------+-------------+--------------+-----------+--------------+------------+---------+--------+---------+--------------+-------------------+---------------------------------+------------------------+-------------------------------+----------------------+--------------------------+----------------------------+-------------------+-------------------+----------------------------+-------------------------+----------------+-----------------+---------------------+--------+
|               legId|searchDate|flightDate|startingAirport|destinationAirport|fareBasisCode|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|baseFare|totalFare|seatsRemaining|totalTravelDistance|segmentsDepartureTimeEpochSeconds|segmentsDepartureTimeRaw|segmentsArrivalTimeEpochSeconds|segmentsArrivalTimeRaw|segmentsArrivalAirportCode|segmentsDepartureAirportCode|segmentsAirlineName|segmentsAirlineCode|segmentsEquipmentDescription|segme

In [ ]:
from pyspark.sql.functions import size

df_no_null_fixed_duration = df_no_null_fixed_duration.withColumn(
    "numStops",
    size(split("segmentsDepartureTimeRaw", r"\|\|")) - 1
)

In [112]:
from pyspark.sql.functions import split

df_no_null_fixed_duration = df_no_null_fixed_duration.withColumn(
    "firstDepartureRaw",
    split("segmentsDepartureTimeRaw", r"\|\|")[0]
)

from pyspark.sql.functions import to_timestamp

df_no_null_fixed_duration = df_no_null_fixed_duration.withColumn(
    "firstDepartureTimestamp",
    to_timestamp("firstDepartureRaw")
)

df_no_null_fixed_duration = df_no_null_fixed_duration.withColumn(
    "lastArrivalTimestamp",
    to_timestamp(
        element_at(
            split("segmentsArrivalTimeRaw", r"\|\|"),
            -1
        )
    )
)

In [114]:
df_no_null_fixed_duration = df_no_null_fixed_duration.drop("firstDepartureRaw")

In [115]:
df_no_null_fixed_duration.show(1)

+--------------------+----------+----------+---------------+------------------+-------------+--------------+-----------+--------------+------------+---------+--------+---------+--------------+-------------------+---------------------------------+------------------------+-------------------------------+----------------------+--------------------------+----------------------------+-------------------+-------------------+----------------------------+-------------------------+----------------+-----------------+---------------------+--------+-----------------------+--------------------+
|               legId|searchDate|flightDate|startingAirport|destinationAirport|fareBasisCode|travelDuration|elapsedDays|isBasicEconomy|isRefundable|isNonStop|baseFare|totalFare|seatsRemaining|totalTravelDistance|segmentsDepartureTimeEpochSeconds|segmentsDepartureTimeRaw|segmentsArrivalTimeEpochSeconds|segmentsArrivalTimeRaw|segmentsArrivalAirportCode|segmentsDepartureAirportCode|segmentsAirlineName|segmentsA

In [121]:
df_no_null_fixed_duration.columns

['legId',
 'searchDate',
 'flightDate',
 'startingAirport',
 'destinationAirport',
 'fareBasisCode',
 'travelDuration',
 'elapsedDays',
 'isBasicEconomy',
 'isRefundable',
 'isNonStop',
 'baseFare',
 'totalFare',
 'seatsRemaining',
 'totalTravelDistance',
 'segmentsDepartureTimeEpochSeconds',
 'segmentsDepartureTimeRaw',
 'segmentsArrivalTimeEpochSeconds',
 'segmentsArrivalTimeRaw',
 'segmentsArrivalAirportCode',
 'segmentsDepartureAirportCode',
 'segmentsAirlineName',
 'segmentsAirlineCode',
 'segmentsEquipmentDescription',
 'segmentsDurationInSeconds',
 'segmentsDistance',
 'segmentsCabinCode',
 'travelDurationMinutes',
 'numStops']

In [119]:
df_no_null_fixed_duration = df_no_null_fixed_duration.drop("firstDepartureTimestamp", "lastArrivalTimestamp")

In [129]:
df_no_null_fixed_duration.select("segmentsDepartureTimeRaw", "segmentsArrivalTimeRaw", "segmentsDurationInSeconds", "travelDurationMinutes", "segmentsAirlineName").limit(100).coalesce(1).write.mode("overwrite").option("header", True).csv("temp.csv")

26/03/03 12:21:27 ERROR Utils: Aborting task 1) / 1][Stage 72:>   (0 + 1) / 1]
scala.MatchError: java.lang.OutOfMemoryError: Java heap space (of class java.lang.OutOfMemoryError)
	at org.apache.spark.sql.execution.datasources.v2.FileDataSourceV2$.attachFilePath(FileDataSourceV2.scala:127)
	at org.apache.spark.sql.execution.datasources.FileScanRDD$$anon$1.hasNext(FileScanRDD.scala:141)
	at scala.collection.Iterator$$anon$9.hasNext(Iterator.scala:593)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage1.processNext(Unknown Source)
	at org.apache.spark.sql.execution.BufferedRowIterator.hasNext(BufferedRowIterator.java:43)
	at org.apache.spark.sql.execution.WholeStageCodegenEvaluatorFactory$WholeStageCodegenPartitionEvaluator$$anon$1.hasNext(WholeStageCodegenEvaluatorFactory.scala:50)
	at scala.collection.Iterator$$anon$10.hasNext(Iterator.scala:610)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodegenStage2.proc

In [176]:
df_final = df_no_null_fixed_duration

In [177]:
df_final.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [ ]:
df_final = df_final.withColumn(
    "segmentsArrivalTime",
    expr("""
        transform(
            split(segmentsArrivalTimeRaw, '\\\\|\\\\|'),
            x -> date_trunc('second', to_timestamp(x))
        )
    """)
)

In [187]:
from pyspark.sql.functions import split

df_final = df_final.withColumn(
    "segmentsCabinCodes",
    split("segmentsCabinCode", r"\|\|")
)

In [186]:
df_final = df_final.withColumn(
    "segmentsDistances",
    expr("""
        transform(
            split(segmentsDistance, '\\\\|\\\\|'),
            x -> cast(x as int)
        )
    """)
)

In [171]:
df_final.segmentsDurationInSeconds = df_no_null_fixed_duration.segmentsDurationInSeconds

In [ ]:
cols = [
    "segmentsDepartureTimeRaw", 
    "segmentsArrivalTimeRaw",
    
    "segmentsArrivalAirportCode",
    "segmentsDepartureAirportCode",
    "segmentsAirlineName",
    "segmentsAirlineCode",
    "segmentsEquipmentDescription",
    "segmentsDurationInSeconds",
    "segmentsDistance",
    "segmentsCabinCode"
]

In [199]:
from pyspark.sql.functions import expr

df_final = df_final.withColumn(
    "layoverDurationMinutes",
    expr("""
        cast(
            travelDurationMinutes -
            (aggregate(segmentsDurationSeconds, 0, (acc, x) -> acc + x) / 60)
        as int)
    """)
)

In [203]:
df_final.printSchema()

root
 |-- legId: string (nullable = true)
 |-- searchDate: date (nullable = true)
 |-- flightDate: date (nullable = true)
 |-- startingAirport: string (nullable = true)
 |-- destinationAirport: string (nullable = true)
 |-- fareBasisCode: string (nullable = true)
 |-- travelDuration: string (nullable = true)
 |-- elapsedDays: integer (nullable = true)
 |-- isBasicEconomy: boolean (nullable = true)
 |-- isRefundable: boolean (nullable = true)
 |-- isNonStop: boolean (nullable = true)
 |-- baseFare: double (nullable = true)
 |-- totalFare: double (nullable = true)
 |-- seatsRemaining: integer (nullable = true)
 |-- totalTravelDistance: integer (nullable = true)
 |-- segmentsDepartureTimeEpochSeconds: string (nullable = true)
 |-- segmentsDepartureTimeRaw: string (nullable = true)
 |-- segmentsArrivalTimeEpochSeconds: string (nullable = true)
 |-- segmentsArrivalTimeRaw: string (nullable = true)
 |-- segmentsArrivalAirportCode: string (nullable = true)
 |-- segmentsDepartureAirportCode: s

In [213]:
df_final.select("segmentsEquipmentDescriptions").show(1)

+-----------------------------+
|segmentsEquipmentDescriptions|
+-----------------------------+
|                [Airbus A321]|
+-----------------------------+
only showing top 1 row


In [205]:
df_clean = df_final.select(
    "legId",
    "searchDate",
    "flightDate",
    "startingAirport",
    "destinationAirport",
    "numStops",
    "layoverDurationMinutes",
    "totalTravelDistance",
    "travelDurationMinutes",
    "seatsRemaining",
    "isBasicEconomy",
    "isRefundable",
    "isNonStop",
    "fareBasisCode",
    "baseFare",
    "totalFare",
    "segmentsDepartureTime",
    "segmentsArrivalTime",
    'segmentsArrivalAirportCodes',
    'segmentsDepartureAirportCodes',
    'segmentsAirlineNames',
    'segmentsAirlineCodes',
    'segmentsEquipmentDescriptions',
    'segmentsDurationSeconds',
    'segmentsDistances',
    'segmentsCabinCodes'
)

In [206]:
df_clean.show()

+--------------------+----------+----------+---------------+------------------+--------+----------------------+-------------------+---------------------+--------------+--------------+------------+---------+-------------+--------+---------+---------------------+--------------------+---------------------------+-----------------------------+--------------------+--------------------+-----------------------------+-----------------------+-----------------+------------------+
|               legId|searchDate|flightDate|startingAirport|destinationAirport|numStops|layoverDurationMinutes|totalTravelDistance|travelDurationMinutes|seatsRemaining|isBasicEconomy|isRefundable|isNonStop|fareBasisCode|baseFare|totalFare|segmentsDepartureTime| segmentsArrivalTime|segmentsArrivalAirportCodes|segmentsDepartureAirportCodes|segmentsAirlineNames|segmentsAirlineCodes|segmentsEquipmentDescriptions|segmentsDurationSeconds|segmentsDistances|segmentsCabinCodes|
+--------------------+----------+----------+--------

In [207]:
df_clean.limit(10).write.mode("overwrite").csv("temp.csv")

AnalysisException: [UNSUPPORTED_DATA_TYPE_FOR_DATASOURCE] The CSV datasource doesn't support the column `segmentsDepartureTime` of the type "ARRAY<TIMESTAMP>". SQLSTATE: 0A000

In [208]:
df_sample = df_clean.sample(
    withReplacement=False,
    fraction=0.03,
    seed=42
)

In [210]:
df_sample.write.mode("overwrite").parquet("flights_clean_1gb")

26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/03 14:27:55 WARN MemoryManager: Total allocation exceeds 95.

In [211]:
df_sample2 = df_clean.sample(
    fraction=0.01,
    seed=1,
    withReplacement=False
)

In [212]:
df_sample2.write.mode("overwrite").parquet("flights_clean_300mb")

26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/03 14:34:32 WARN MemoryManager: Total allocation exceeds 95.

26/03/03 14:47:19 WARN BasicWriteTaskStatsTracker: Expected 1 files, but only saw 0. This could be due to the output format not writing empty files, or files being not immediately visible in the filesystem.
